In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ale_py import ALEInterface, roms
import random

In [ ]:
ale = ALEInterface()
ale.setInt("random_seed", 42)
ale.setFloat("repeat_action_probability", 0.0)
ale.setBool("color_averaging", False)
ale.loadROM(roms.get_rom_path("beam_rider"))
actions = ale.getMinimalActionSet()
ale.reset_game()
frames = []
# Take a few random steps to get past the title/start screen
for _ in range(60):
    ale.act(random.choice(actions))
    if ale.game_over():
        ale.reset_game()
# Collect 8 frames spread out over gameplay
for i in range(8):
    for _ in range(15):  # skip ahead between captures
        ale.act(random.choice(actions))
        if ale.game_over():
            ale.reset_game()
    buf = np.zeros((210, 160, 3), dtype=np.uint8)
    ale.getScreenRGB(buf)
    frames.append(buf.copy())
print(f"Captured {len(frames)} frames, shape: {frames[0].shape}")  # (210, 160, 3)


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 9))
for ax, frame in zip(axes.flat, frames):
    ax.imshow(frame)
    ax.set_title(f"Full frame {frame.shape}")
    ax.axis("off")
plt.suptitle("Raw Beam Rider frames (210×160)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
frame = frames[2]
fig, ax = plt.subplots(figsize=(8, 11))
ax.imshow(frame)
ax.set_title("Beam Rider – pixel coordinate guide", fontsize=13)
# horizontal gridlines every 10 rows
for y in range(0, 210, 10):
    ax.axhline(y, color="cyan", linewidth=0.4, alpha=0.5)
    ax.text(-4, y, str(y), fontsize=6, color="cyan", va="center", ha="right")
# vertical gridlines every 10 cols
for x in range(0, 160, 10):
    ax.axvline(x, color="lime", linewidth=0.4, alpha=0.5)
    ax.text(x, -3, str(x), fontsize=6, color="lime", ha="center", va="bottom")
plt.tight_layout()
plt.show()


In [ ]:
# Change these once you've decided
ROW_START, ROW_END = 30, 190   # rows to keep
COL_START, COL_END = 8, 160   # cols to keep (full width)
fig, axes = plt.subplots(1, 2, figsize=(12, 7))
# Left: original with crop box overlaid
axes[0].imshow(frame)
rect = patches.Rectangle(
    (COL_START, ROW_START),
    COL_END - COL_START, ROW_END - ROW_START,
    linewidth=2, edgecolor="red", facecolor="none"
)
axes[0].add_patch(rect)
axes[0].set_title(f"Original (red = crop region)", fontsize=11)
axes[0].axis("off")
# Right: the cropped result
cropped = frame[ROW_START:ROW_END, COL_START:COL_END]
axes[1].imshow(cropped)
axes[1].set_title(f"Cropped → {cropped.shape[0]}×{cropped.shape[1]}", fontsize=11)
axes[1].axis("off")
plt.suptitle(f"frame[{ROW_START}:{ROW_END}, {COL_START}:{COL_END}]", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import torchvision.transforms as T
import torch
# Your confirmed crop from notes.md
ROW_START, ROW_END = 40, 190
COL_START, COL_END = 30, 140
# Sizes to try (H, W) — feel free to add more
TARGET_SIZES = [84, 72, 64, 56, 48, 40, 32]
def make_preprocessor(h, w):
    return T.Compose([
        T.ToPILImage(),
        T.Grayscale(),
        T.Resize((h, w), interpolation=T.InterpolationMode.BILINEAR),
        T.PILToTensor(),
    ])
# Use the frame you inspected earlier (frames[3] or pick any)
sample_frame = frames[3]
cropped_rgb = sample_frame[ROW_START:ROW_END, COL_START:COL_END]  # (150, 110, 3)
print(f"Crop shape: {cropped_rgb.shape}  ({ROW_END-ROW_START}×{COL_END-COL_START} px)")
# --- Plot grid ---
n = len(TARGET_SIZES)
fig, axes = plt.subplots(1, n, figsize=(3 * n, 4))
for ax, size in zip(axes, TARGET_SIZES):
    tf = make_preprocessor(size, size)
    gray = tf(cropped_rgb).squeeze(0).numpy()   # (H, W) uint8
    bytes_u8   = gray.nbytes
    bytes_u4   = gray.nbytes // 2               # notional 4-bit
    ax.imshow(gray, cmap="gray", interpolation="nearest")
    ax.set_title(
        f"{size}×{size}\n"
        f"u8: {bytes_u8} B\n"
        f"u4: {bytes_u4} B",
        fontsize=9
    )
    ax.axis("off")
plt.suptitle(
    f"Grayscale compression sweep  |  crop [{ROW_START}:{ROW_END}, {COL_START}:{COL_END}]\n"
    "Same ToPILImage→Grayscale→Resize→PILToTensor pipeline as dqn.py",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.show()
# ── Byte cost for a 1M-frame replay buffer at each size ──────────────────────
print("\n── Replay buffer memory cost (1M frames, uint8) ──")
print(f"{'Size':>8}  {'Per frame':>12}  {'1M frames':>12}  {'1M frames (GB)':>16}")
print("-" * 56)
for size in TARGET_SIZES:
    per_frame = size * size          # uint8 = 1 byte/pixel
    total_mb  = per_frame * 1_000_000 / 1024**3
    print(f"{size:>4}×{size:<4}  {per_frame:>10} B  {per_frame*1_000_000:>12,}  {total_mb:>14.2f} GB")
